# Game Data Analysis

This notebook analyzes the crawled game data by:
1. Loading all games from CSV
2. Removing duplicate games (by title)
3. Filtering games with >=10k reviews and >=4.0 rating
4. Analyzing the remaining dataset


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Set up plotting style
plt.style.use('default')
sns.set_palette("husl")

print("Libraries imported successfully!")


## 1. Load the CSV Data


In [ ]:
# Load the CSV file
csv_path = Path('/Users/lance/Documents/GitHub/aigamestore/crawled_games/processed_data/all_games_data.csv')
df = pd.read_csv(csv_path)

print(f"Loaded {len(df)} games from CSV")
print(f"Columns: {list(df.columns)}")
print(f"\nDataFrame shape: {df.shape}")
print(f"\nFirst few rows:")
df.head()


## 2. Initial Data Overview


In [ ]:
# Basic statistics
print("=== INITIAL DATA OVERVIEW ===")
print(f"Total games: {len(df):,}")
print(f"Unique games (by title): {df['game_name'].nunique():,}")
print(f"Games with ratings > 0: {len(df[df['rating'] > 0]):,}")
print(f"Games with reviews > 0: {len(df[df['reviews'] > 0]):,}")
print(f"Games with >= 10k reviews: {len(df[df['reviews'] >= 10000]):,}")
print(f"Games with >= 4.0 rating: {len(df[df['rating'] >= 4.0]):,}")

# Rating and review statistics
print(f"\n=== RATING STATISTICS ===")
print(f"Average rating: {df['rating'].mean():.2f}")
print(f"Median rating: {df['rating'].median():.2f}")
print(f"Min rating: {df['rating'].min():.2f}")
print(f"Max rating: {df['rating'].max():.2f}")

print(f"\n=== REVIEW STATISTICS ===")
print(f"Average reviews: {df['reviews'].mean():,.0f}")
print(f"Median reviews: {df['reviews'].median():,.0f}")
print(f"Min reviews: {df['reviews'].min():,}")
print(f"Max reviews: {df['reviews'].max():,}")


## 3. Remove Duplicate Games (by title)


In [ ]:
# Check for duplicates before removal
duplicate_count = len(df) - df['game_name'].nunique()
print(f"Number of duplicate games (by title): {duplicate_count:,}")

# Remove duplicates, keeping the first occurrence
df_no_duplicates = df.drop_duplicates(subset=['game_name'], keep='first')

print(f"\nAfter removing duplicates:")
print(f"Games remaining: {len(df_no_duplicates):,}")
print(f"Games removed: {len(df) - len(df_no_duplicates):,}")

# Show some examples of games that had duplicates
if duplicate_count > 0:
    duplicate_games = df[df.duplicated(subset=['game_name'], keep=False)].sort_values('game_name')
    print(f"\nExample games with duplicates:")
    print(duplicate_games[['game_name', 'country', 'genre', 'rating', 'reviews']].head(10))


## 4. Filter Games: >=10k Reviews and >=4.0 Rating


In [ ]:
# Apply filters
print("=== APPLYING FILTERS ===")
print(f"Filter 1: Games with >= 10,000 reviews")
df_high_reviews = df_no_duplicates[df_no_duplicates['reviews'] >= 10000]
print(f"Games after review filter: {len(df_high_reviews):,}")

print(f"\nFilter 2: Games with >= 4.0 rating")
df_filtered = df_high_reviews[df_high_reviews['rating'] >= 4.0]
print(f"Games after rating filter: {len(df_filtered):,}")

print(f"\n=== FILTERING SUMMARY ===")
print(f"Original games: {len(df):,}")
print(f"After removing duplicates: {len(df_no_duplicates):,}")
print(f"After review filter (>=10k): {len(df_high_reviews):,}")
print(f"After rating filter (>=4.0): {len(df_filtered):,}")
print(f"\nFinal games remaining: {len(df_filtered):,}")


## 5. Analysis of Filtered Games


In [ ]:
if len(df_filtered) > 0:
    print("=== FILTERED GAMES ANALYSIS ===")
    
    # Genre distribution
    print(f"\nGenre distribution:")
    genre_counts = df_filtered['genre'].value_counts()
    for genre, count in genre_counts.items():
        print(f"  {genre}: {count} games")
    
    # Country distribution
    print(f"\nTop 10 countries by game count:")
    country_counts = df_filtered['country'].value_counts().head(10)
    for country, count in country_counts.items():
        print(f"  {country}: {count} games")
    
    # Rating and review statistics for filtered games
    print(f"\nRating statistics (filtered games):")
    print(f"  Average rating: {df_filtered['rating'].mean():.2f}")
    print(f"  Median rating: {df_filtered['rating'].median():.2f}")
    print(f"  Min rating: {df_filtered['rating'].min():.2f}")
    print(f"  Max rating: {df_filtered['rating'].max():.2f}")
    
    print(f"\nReview statistics (filtered games):")
    print(f"  Average reviews: {df_filtered['reviews'].mean():,.0f}")
    print(f"  Median reviews: {df_filtered['reviews'].median():,.0f}")
    print(f"  Min reviews: {df_filtered['reviews'].min():,}")
    print(f"  Max reviews: {df_filtered['reviews'].max():,}")
    
else:
    print("No games meet the criteria (>=10k reviews and >=4.0 rating)")


## 6. Top Games by Rating and Reviews


In [ ]:
if len(df_filtered) > 0:
    print("=== TOP GAMES ===")
    
    # Top 10 games by rating
    print(f"\nTop 10 games by rating:")
    top_rated = df_filtered.nlargest(10, 'rating')[['game_name', 'genre', 'country', 'rating', 'reviews']]
    for idx, row in top_rated.iterrows():
        print(f"  {row['game_name']} - {row['rating']:.1f}★ ({row['reviews']:,} reviews) [{row['genre']}, {row['country']}]")
    
    # Top 10 games by review count
    print(f"\nTop 10 games by review count:")
    top_reviewed = df_filtered.nlargest(10, 'reviews')[['game_name', 'genre', 'country', 'rating', 'reviews']]
    for idx, row in top_reviewed.iterrows():
        print(f"  {row['game_name']} - {row['reviews']:,} reviews ({row['rating']:.1f}★) [{row['genre']}, {row['country']}]")
    
    # Games with perfect 5.0 rating
    perfect_games = df_filtered[df_filtered['rating'] == 5.0]
    if len(perfect_games) > 0:
        print(f"\nGames with perfect 5.0 rating ({len(perfect_games)} games):")
        for idx, row in perfect_games.iterrows():
            print(f"  {row['game_name']} - {row['reviews']:,} reviews [{row['genre']}, {row['country']}]")
    else:
        print(f"\nNo games with perfect 5.0 rating found.")


## 7. Summary and Conclusions


In [ ]:
print("=== FINAL SUMMARY ===")
print(f"📊 Original dataset: {len(df):,} games")
print(f"🔄 After removing duplicates: {len(df_no_duplicates):,} games")
print(f"⭐ After filtering (>=10k reviews & >=4.0 rating): {len(df_filtered):,} games")

if len(df_filtered) > 0:
    print(f"\n🎯 Key Findings:")
    print(f"   • {len(df_filtered):,} games meet the high-quality criteria")
    print(f"   • Most popular genre: {df_filtered['genre'].mode().iloc[0] if len(df_filtered) > 0 else 'N/A'}")
    print(f"   • Average rating: {df_filtered['rating'].mean():.2f}★")
    print(f"   • Average reviews: {df_filtered['reviews'].mean():,.0f}")
    print(f"   • Highest rated game: {df_filtered.loc[df_filtered['rating'].idxmax(), 'game_name']} ({df_filtered['rating'].max():.1f}★)")
    print(f"   • Most reviewed game: {df_filtered.loc[df_filtered['reviews'].idxmax(), 'game_name']} ({df_filtered['reviews'].max():,} reviews)")
else:
    print(f"\n⚠️  No games meet the strict criteria of >=10k reviews and >=4.0 rating.")
    print(f"   Consider relaxing the criteria or analyzing the data differently.")

print(f"\n✅ Analysis complete!")
